<h1><center>Workshop on Computer Vision for Image Segmentation</center></h1>

Prepared by **Vladimir PIMONOV**  
Affiliation: [ILM / Team PNEC]  
Contact: [foxlightmanstudes@gmail.com]

These notebooks were prepared for the workshop and are intended as guided practical material.  
They can also be used independently outside the workshop.  
For questions, feedback, or bug reports, please contact the author.

<h2><center>PW 01. Preparation of data</center></h2>

In this notebook, we prepare a segmentation dataset for training a computer vision model.

Our goal is to start from full microscopy images and their masks, then build a clean and balanced collection of image patches that can be used for training, validation, and testing.

The main steps are:

1. extract masks from annotation files,
2. match masks with the original images,
3. describe candidate patches in a dataframe,
4. split the dataset into training, validation, and test sets,
5. save the final image and mask patches to disk.

## 1. Import the required libraries

We begin by importing the Python packages used throughout the notebook.

These libraries help us:

- read microscopy images and TIFF files,
- navigate through folders,
- handle tabular data,
- read Photoshop annotation files,
- process masks and connected components,
- display images and results.

At this stage, there is nothing specific to machine learning yet. We are building the data preparation pipeline first.

In [ ]:
import tifffile
from tifffile import TiffFile

import matplotlib.pyplot as plt

import numpy as np
import os
from os.path import join as pjoin

### Select correct progressbar widget depending on the system used for environment
if "VSCODE_PID" in os.environ:
    from tqdm import tqdm
else:
    from tqdm.notebook import tqdm
import gc

from PIL import Image

import pandas as pd

from psd_tools import PSDImage

from skimage.measure import label, regionprops

<h3><center>Extraction of Masks from .psd files</center></h3>

## 2. Define where the raw data is stored

Here we choose the main folder that contains the images and annotation files.

Keeping a single root path makes the rest of the notebook easier to manage, because all file searches and output folders will be built from this location.

In [ ]:
### 1. select the root folder with your data

### Or as a relative path
root = r"Agregates for training TP"

## 3. Find the annotation files

The annotations are stored in `.psd` files.  
In this step, we search the dataset folder recursively and collect all files of that type.

The search is done with `os.walk`, which is a standard Python tool for exploring a directory tree.  
It goes through the chosen folder and, for each level, returns:

- the current folder path,
- the list of subfolders,
- the list of files found there.

This makes it possible to search not only in one folder, but also in all nested subfolders automatically.

The code uses a **list comprehension** to build the list of annotation files in a compact way.  
A list comprehension is a short Python syntax used to create a list directly from a loop and, if needed, from a condition.

Here it is useful because it allows us to say, in one expression:

- go through all folders returned by `os.walk`,
- look at all files inside them,
- keep only the ones whose name ends with `.psd`,
- store their full paths in a list.

This gives us the list of annotated samples from which the segmentation masks will be extracted.

In [ ]:
### 2. Search for the necessary type of files

### 2.1 Define file type for search
file_type = '.psd'

### 2.2 Use list comprehension to make a list of files
#       os.walk skim throug the system while and gathers
#       the paths of files eding with the .psd extension
psd_files = [os.path.join(r,file) for r,d,f in os.walk(root) for file in f if 
             file.lower().endswith(file_type)
            ]

### 2.3 Have a look at the data amount we have
print("We found %.i files in total" % len(psd_files))

## 4. Extract segmentation masks from the annotation files

Now we go through each `.psd` file and convert the annotation layer into a mask saved as a TIFF image.

The purpose of this step is to transform the manual annotations into a format that is easy to use later in Python and PyTorch.

A few important checks are included here:

- we verify that the expected annotation layer exists,
- we place the mask back onto the full canvas when needed,
- we save all masks in a consistent format.

After this step, the dataset contains explicit image files and explicit mask files, which is much more convenient for training.

In [ ]:
### 3. Go file by file, extract mask from it and save it as a separate RGB .tiff file

for psd_path in tqdm(psd_files[:]):
    
    ### 3.1 split the file path and file title
    file_path, file_name_with_extention = os.path.split(psd_path)
    file_name, file_extention = os.path.splitext(file_name_with_extention)
    
    ### 3.2 Create the folder for masks in the same directory as the .psd files
    saving_address_mask = pjoin(file_path, 'Masks')

    if not os.path.exists(saving_address_mask):
        os.makedirs(saving_address_mask)
    else:    
        pass
    
    ### 3.3 Read the .psd file we created
    img = PSDImage.open(psd_path)
    
    ### 3.4 Verify that the file contain the layer with nanoparticles "NPs"
    #       or with binary mask "bin"
    #       NB! assert will raise an error is the assertion is condition is not met
    assert len([layer.name.lower() for layer in img if 'bin' in layer.name.lower() or 'NPs' in layer.name])==1
    
    ### 3.5 Go layer by layer and save it as a mask with red images
    for layer in img:
        
        layer.visible = True
        
        if 'bin' in layer.name.lower():
            
            np_mask = np.array(layer.composite().convert("RGB"))
            
            if np_mask.sum() > ~np_mask.sum():
                raise
            
            ### make red mask to save it as the other ones
            np_mask = np_mask * np.array([1, 0, 0])[np.newaxis,np.newaxis,:]
            
            tifffile.imwrite(pjoin(saving_address_mask, f'{file_name}_MASK.tiff'), np.uint8(np_mask))
            
        elif 'nps' in layer.name.lower():
            
            np_mask = np.array(layer.composite().convert("RGB"))
            
            # tranparent masks opens with white filling
            # replace it with black
            white = np.all(np_mask == np.array([255, 255, 255], dtype=np_mask.dtype), axis=-1)
            np_mask[white] = [0, 0, 0]
            
            # Find the offset of the background mask
            x_off, y_off = layer.offset
            
            # Get full PSD canvas size
            psd_w, psd_h = img.size
            
            # Make the empty mask
            full_mask = np.zeros((psd_h, psd_w, 3), dtype=np.uint8)
            
            # Fill the empty mask with the image mask
            h, w, _ = np_mask.shape
            full_mask[y_off:y_off+h, x_off:x_off+w, :] = np_mask
            
            tifffile.imwrite(pjoin(saving_address_mask, f'{file_name}_MASK.tiff'), np.uint8(full_mask))

## 5. Match the original images with their masks

Once the masks have been extracted, we collect two file lists:

- the microscopy images,
- the corresponding mask images.

The idea is to build the dataset from paired samples, where each raw image has one associated segmentation mask.

Some folders or image types can also be excluded here if they should not be used for training.

In [ ]:
### Serach for the masks and for the tiff files

images = [os.path.join(r,file) for r,d,f in os.walk(root)
          for file in f
          if file.lower().endswith('.tiff') and 'mask' not in file.lower()
          and 'Co150V' not in r and 'Training' not in r
         ]

masks = [os.path.join(r,file) for r,d,f in os.walk(root)
          for file in f
          if file.lower().endswith('_mask.tiff')
          and 'Co150V' not in r and 'Training' not in r
        ]

print("Found total of %.i Masks and %.i Images" %(len(masks), len(images)))

## 6. Define the patching strategy

The full microscopy images do not all have the same magnification, and the nanoparticles do not all appear at the same apparent size.

Because of that, we do not want to crop every image in exactly the same way.  
Instead, we define, for each magnification group, a list of **patch size multipliers**.

This means that images acquired at a given magnification will first be cut into patches whose size depends on that magnification.  
These larger patches are then **shrunk back to one common base patch size** before training.

This strategy has several goals.

### 6.1. Keep the training size fixed

Neural networks are easier and more efficient to train when all input patches have the same final size.  
Using one common base patch size simplifies batching, memory management, and model training.

### 6.2. Adapt the effective field of view to the magnification

If two images have different magnifications, the same fixed crop size in pixels does not correspond to the same physical scale.  
By adjusting the crop size before resizing, we can present the network with image patches that contain a more comparable structural context across different magnifications.

This helps the model remain more stable when trained on images acquired at different scales or containing nanoparticles of different apparent sizes.

### 6.3. Use only shrinkage, not expansion

An important point is that we only use patch sizes that will later be **shrunk** to the base patch size, not expanded.

In other words, we avoid taking small image patches and enlarging them artificially.  
This is done to avoid training the model on interpolated details that were not present in the original data.

Shrinking a larger crop preserves real information while standardizing the final input size.  
Expanding a smaller crop would create artificial pixel values and could teach the network to rely on interpolation artifacts rather than on real image structures.

### 6.4. Increase the variability of useful training examples

This patching strategy also makes it possible to broaden the range of effective magnifications seen during training.  
As a result, the model can learn from a more diverse set of visual scales while still receiving inputs in a consistent format.

#### Manual size measurements and patch characterization

For each magnification, we manually measured the average apparent diameter of the nanoparticles.  
These measurements are then used as a reference when characterizing the extracted patches.

In particular, they help define whether a patch should be considered:

- **empty**,
- **almost empty**,
- or clearly containing an object.

Here, **empty** does not necessarily mean perfectly empty in the literal sense.  
It means **functionally empty**: the amount of annotated object pixels is so small compared with the background that, for training purposes, the patch behaves essentially like a background patch.

This distinction is useful because in real microscopy data, a patch may contain a tiny fragment or negligible trace of an object while still being practically equivalent to empty background for the segmentation task.

In [ ]:
### dictionary describing the patching of the images with different magnifications with different particle sizes
### the magnification keys will be used to calculate the threshold for the mask to be concidered functionally empty
### that is having less than 1 np per image
### This control is necessary to avoid overpopulation of the dataset with empty images
### make pathes of different sizes to avoid heavily over weight examples

patch_dict_512 = {'4nm' : {'30000X_25' : [1, 1.5, 2], # average diameter of particles 25pix
                       '60000X_45' : [1, 2, 3], # average diameter of particles 55pix
                       '100000X_70' : [1, 2, 3], # average diameter of particles 70pix
                       '150000X_100' : [1, 2, 3] # average diameter of particles 100pix
                      },
              '2.5nm' : {'60000X_30' : [1, 1.5, 2], # average diameter of particles 25pix
                         '100000X_40' : [1, 2, 3], # average diameter of particles 40pix
                         '150000X_70' : [1, 2, 3], # average diameter of particles 70pix
                        },
              'FeCo' : {'120k_15' : [1, 1.5, 2] # average diameter of particles 15pix
                       },
              'FePt' : {'200k_20' : [1, 1.5, 2] # average diameter of particles 20pix
                       },
             }

patch_dict_256 = {'4nm' : {'30000X_25' : [1, 1.5, 2], # average diameter of particles 25pix
                           '60000X_45' : [1, 1.5, 2, 2.5], # average diameter of particles 55pix
                           '100000X_70' : [1.5, 2, 2.5], # average diameter of particles 70pix
                           '150000X_100' : [2, 2.5, 3] # average diameter of particles 100pix
                          },
                  '2.5nm' : {'60000X_30' : [1, 1.5, 2], # average diameter of particles 25pix
                             '100000X_40' : [1, 1.5, 2, 2.5], # average diameter of particles 40pix
                             '150000X_70' : [1.5, 2, 2.5], # average diameter of particles 70pix
                            },
                  'FeCo' : {'120k_15' : [1, 1.5, 2] # average diameter of particles 15pix
                           },
                  'FePt' : {'200k_20' : [1, 1.5, 2] # average diameter of particles 20pix
                           },
                 }

patch_dict_128 = {'4nm' : {'30000X_25' : [1, 1.5], # average diameter of particles 25pix
                           '60000X_45' : [1, 1.5, 2], # average diameter of particles 55pix
                           '100000X_70' : [1.5, 2], # average diameter of particles 70pix
#                            '150000X_100' : [2, 2.5] # average diameter of particles 100pix
                          },
                  '2.5nm' : {'60000X_30' : [1, 1.5], # average diameter of particles 25pix
                             '100000X_40' : [1, 1.5, 2], # average diameter of particles 40pix
                             '150000X_70' : [1.5, 2], # average diameter of particles 70pix
                            },
                  'FeCo' : {'120k_15' : [1, 1.5] # average diameter of particles 15pix
                           },
                  'FePt' : {'200k_20' : [1, 1.5] # average diameter of particles 20pix
                           },
                 }

## 7. Define how patches are placed on an image

To extract patches from a large image, we need to decide where each crop starts.

This helper function computes a list of starting positions along one dimension.  
It ensures that:

- patches can overlap,
- the full image is covered,
- the last patch still reaches the border.

This is important because we do not want to systematically ignore the edges of the images.

In [ ]:
### define functions for patching

def compute_starts(length: int, patch: int, overlap: float = 0.5):
    if patch > length:
        # either return [0] and later pad, or handle separately
        return np.array([0], dtype=int)

    stride = max(1, int(round(patch * (1 - overlap))))  # overlap=0.5 -> stride ~ patch/2
    starts = np.arange(0, length - patch + 1, stride, dtype=int)

    last = length - patch
    if starts.size == 0:
        starts = np.array([0], dtype=int)
    if starts[-1] != last:
        starts = np.append(starts, last)

    # optional: ensure unique + sorted (defensive)
    starts = np.unique(starts)
    return starts

## 8. Describe all candidate patches in a dataframe

This is the central dataset-description step.

At this stage, we still do **not** save the cropped patches themselves.  
Instead, we go through all source images and masks, generate all candidate crop positions for all selected patch sizes, and record their properties in a dataframe.

For each candidate patch, we store information such as:

- which original image it comes from,
- which mask it is associated with,
- the patch position in the source image,
- the patch size or multiplier used,
- the magnification group,
- and several indicators describing the mask content inside that patch.

In particular, we analyze how much annotated object is present in the patch and whether it should be classified as:

- **empty**,
- **almost empty**,
- or clearly containing a meaningful object region.

These categories are defined using the previously measured average nanoparticle size for each magnification.  
This gives us practical thresholds, or **caps**, to decide when the amount of annotated material is negligible and when it becomes relevant for training.

### Why build the dataframe first?

This intermediate dataframe is extremely useful because, as we will see below, the total number of candidate patches can be very large, and many of them are **empty** or **almost empty**.

In some images, these low-content patches can dominate the dataset very strongly.  
If we generated and saved all patches immediately, we would produce a large amount of data that would later be discarded or heavily downsampled.

By first describing all candidates in a dataframe, we can:

- inspect the dataset composition,
- quantify how many empty and almost empty patches exist,
- define balancing rules,
- and only then generate the subset of patches that is actually useful.

This makes the pipeline much more efficient, both in disk usage and in preprocessing time.

### Why define empty and almost empty caps?

The goal is not only to label the patches, but also to prepare for **dataset balancing**.

If empty or almost empty patches are too numerous, the model may see too much background during training and not enough informative object examples.  
Defining explicit caps for these categories allows us to control their proportion later, so that the final training set remains more balanced and more useful for learning.

In short, the dataframe serves as a structured description of all possible training candidates.  
It allows us to analyze and filter the dataset before committing to the much heavier step of writing all cropped patches to disk.

In [ ]:
### create the dataframe with selection of the empty images

### Select parameters of the patching
base_patch_size = 512
overlap = 1/2

### Select the parameters of emptiness of the patches
empty_cap = 1
almost_empty_cap = 3

df = pd.DataFrame([])

### Below is just a fancy way to select one of the patch dictionaries usitg
### the value of base patch size
patch_dict = eval(f"patch_dict_{base_patch_size}")

for np_size in tqdm(patch_dict.keys()):
    
    for mag_diam in patch_dict[np_size].keys():
        
        print(np_size, mag_diam)
        
        mag, diam = mag_diam.split('_')
        
        patch_sizes = patch_dict[np_size][mag_diam]
        
        masks_mag = [i for i in masks if np_size in i and mag in i]
        
        for mask_path in tqdm(masks_mag):
            
            file_name = mask_path.split('\\')[-1].split('.tiff')[0]
            
            with TiffFile(mask_path) as tif:
                mask = tif.asarray()
                ### binarize the mask
                mask_bin = (mask.sum(axis = 2)>128).astype(int)
                
            h, w = mask_bin.shape
            
            for patch_mult in patch_sizes:
                
                ps = base_patch_size*patch_mult
                
                ### normalize the diameter in order to calculate the treshold area coherently
                ### with the area of occupied pixels
                norm_diam = float(diam)/patch_mult
                tresh_area = np.pi*(float(norm_diam)/2)**2
                
                touch_tol = int(min(norm_diam/2, 25))
                touch_out = base_patch_size - 1 - touch_tol
                
                w_starts = compute_starts(w, ps, overlap).astype(int)
                h_starts = compute_starts(h, ps, overlap).astype(int)
                
                for h_st in h_starts:
                    for w_st in w_starts:
                        ### make mask patches
                        mask_patch = mask_bin[h_st:int(h_st+ps), w_st:int(w_st+ps)]
                        
                        mask_image = Image.fromarray(mask_patch)
                        mask_image = mask_image.resize((base_patch_size,base_patch_size),
                                                       resample = Image.NEAREST)
                        
                        mask_patch_res = np.array(mask_image)
                        
                        pos_pix = mask_patch_res.sum()
                        
                        if pos_pix > 0:
                            
                            lab_mask = label(mask_patch_res)
                            
                            touch_border_list = []
                            touch_both_sides_list = []
                            touch_corner_list = []
                            
                            for idx in np.unique(lab_mask)[1:]:
                                
                                sub_mask = (lab_mask==idx)
                            
                                y_s, x_s = np.where(sub_mask!=0)

                                y_min, y_max, x_min, x_max = y_s.min(), y_s.max(), x_s.min(), x_s.max()
                                
                                touches_left   = (x_min <= touch_tol)
                                touches_top    = (y_min <= touch_tol)
                                touches_right  = (x_max >= touch_out)
                                touches_bottom = (y_max >= touch_out)
                                
                                touch_border_list.append(touches_left or touches_top or touches_right or touches_bottom)

                                touch_both_sides_list.append((touches_left and touches_right) or\
                                                                (touches_top and touches_bottom))
                                touch_corner_list.append((touches_left and touches_top) or\
                                                        (touches_left and touches_bottom) or\
                                                        (touches_right and touches_top) or\
                                                        (touches_right and touches_bottom))
                            
                            touch_border = np.sum(touch_border_list)
                            touch_both_sides = np.sum(touch_both_sides_list)
                            touch_corner = np.sum(touch_corner_list)
                            n_partices = lab_mask.max()
                        else:
                            touch_border = 0
                            touch_both_sides = 0
                            touch_corner = 0
                            n_partices = 0                        
                        
                        if df.size==0:
                            df = pd.DataFrame(data = np.array([file_name, np_size,
                                                               mag, diam, patch_mult,
                                                               w_st, h_st,
                                                               int(pos_pix), int(tresh_area),
                                                               touch_border, touch_both_sides,
                                                               touch_corner, n_partices
                                                              ])[np.newaxis,:],
                                              columns = ['File_name', 'NP_size',
                                                         'Magnification', 'Average_diameter', 'Patch_mult',
                                                         'Patch_x_pos', 'Patch_y_pos', 
                                                         'Positive_pixels', 'Negative_treshold',
                                                         'Touch_border', 'Touch_both_sides', 'Touch_corner',
                                                         'N_NPs'])
                        else:
                            
                            sdf = pd.DataFrame(data = np.array([file_name, np_size,
                                                               mag, diam, patch_mult,
                                                               w_st, h_st,
                                                               int(pos_pix), int(tresh_area),
                                                               touch_border, touch_both_sides,
                                                                touch_corner, n_partices
                                                              ])[np.newaxis,:],
                                              columns = ['File_name', 'NP_size',
                                                         'Magnification', 'Average_diameter', 'Patch_mult',
                                                         'Patch_x_pos', 'Patch_y_pos', 
                                                         'Positive_pixels', 'Negative_treshold',
                                                         'Touch_border', 'Touch_both_sides', 'Touch_corner',
                                                         'N_NPs'])
                            df = pd.concat([df, sdf])

saving_path = pjoin(root, f'Training {base_patch_size} TP')

if not os.path.exists(saving_path):
    os.makedirs(saving_path)
else:
    pass

int_cols = ['Average_diameter', 'Patch_x_pos', 'Patch_y_pos',
            'Positive_pixels', 'Negative_treshold', 'Touch_border', 'Touch_both_sides', 'Touch_corner']
float_cols = ['Patch_mult']

for col in int_cols:
    df[col] = df[col].astype(int)
for col in float_cols:
    df[col] = df[col].astype(float)

df.reset_index(inplace = True, drop = True)
df['Empty'] = df['Positive_pixels'].astype(float)<df['Negative_treshold'].astype(float)*empty_cap
df['Almost_Empty'] = (df['Positive_pixels'].astype(float)>=df['Negative_treshold'].astype(float)*empty_cap) &\
                    (df['Positive_pixels'].astype(float)<almost_empty_cap*df['Negative_treshold'].astype(float)) 

df.to_csv(pjoin(saving_path, f'Whole Training set {base_patch_size}.csv'), sep = '\t')

## 9. Inspect the patch metadata

Now we look at the dataframe to understand what has been generated.

At this stage, we are not training a model yet.  
We are checking whether the patch description step makes sense and whether the statistics look reasonable.

This inspection step is very important in computer vision workflows: many training problems actually come from dataset construction issues rather than from the model itself.

In [ ]:
df.T

## 10. Summarize the patch distribution

Here we compute grouped statistics to better understand the composition of the candidate patch set.

At this stage, the goal is not yet to select the final training samples, but to inspect how the generated patches are distributed across:

- magnification groups,
- patch scales,
- empty and almost empty categories,
- and other mask-related descriptors such as border contact.

This inspection is important because it reveals how **imbalanced** the candidate dataset is.

In practice, some image groups produce a very large number of patches containing little or no useful foreground information.  
For certain magnifications, the proportion of **functionally empty** patches can exceed 50% of all candidates.  
In other words, more than half of the possible crops for those images may be dominated by background.

This matters because such an imbalance would strongly affect training if all patches were kept as they are.  
A model trained on too many empty or almost empty examples may learn to predict background too often and may not receive enough informative positive examples.

That is why this statistical summary is a key intermediate step.  
It allows us to measure the imbalance before generating the actual patch files and prepares the next section, where the dataset will be balanced by controlling the proportion of empty and almost empty patches.

In [ ]:
df.groupby(['NP_size', 'Magnification', 'Patch_mult']).apply(lambda x: pd.Series({
    'Touch_border (%)' : ((x['Touch_border']!=0).sum()/x.shape[0]*100).round(1),
    'Touch_both_sides (%)' : ((x['Touch_both_sides']!=0).sum()/x.shape[0]*100).round(1),
    'Touch_corner (%)' : ((x['Touch_corner']!=0).sum()/x.shape[0]*100).round(1),
    'Almost Empty (%)' : ((x['Almost_Empty']!=0).sum()/x.shape[0]*100).round(1),
    'Empty (%)' : ((x['Empty']!=0).sum()/x.shape[0]*100).round(1),
})
                                                            ).reset_index()

## 11. Reserve complete source images for validation and test

To avoid data leakage, we do not split the dataset at the patch level only.

Instead, we first choose complete original images that will be used exclusively for validation and test.  
Then all patches coming from those images are kept out of training.

This is important because nearby patches from the same microscopy image often share texture, noise, and local structure.  
If such patches were split across training and evaluation sets, the validation results would be overly optimistic.

In [ ]:
### select one image per magnification, per nanoparticle size to use all patches from them for validation/test
### it is necessary to avoid any leakage of the textures to the validation and test

test_val = ['Fe_300V_4nm_TM25F08-200000.0V-30000X-pix0.21951[nm]-0002_MASK',
            'Fe_300V_4nm_TM25F08-200000.0V-60000X-pix0.11095[nm]-0005_MASK',
            'Fe_300V_4nm_TM25F08-200000.0V-100000X-pix0.06508[nm]-0021_MASK',
            'Fe_300V_4nm_TM25F08-200000.0V-150000X-pix0.04291[nm]-0030_MASK',
            'Fe_75V_2.5nm_TM25F06-200000.0V-60000X-pix0.11095[nm]-0003_MASK',
            'Fe_75V_2.5nm_TM25F06-200000.0V-100000X-pix0.06508[nm]-0008_MASK',
            'Fe_75V_2.5nm_TM25F06-200000.0V-150000X-pix0.04291[nm]-0019_MASK',
            'Frame_120k-pix0.10787[nm]_10_MASK',
            'Frame_200k-pix0.06526[nm]_13_MASK']

## 12. Build the training subset

Now we define which candidate patches will actually be kept for training.

The goal is to avoid a dataset dominated by a few very frequent categories, especially:

- **empty** patches,
- **almost empty** patches,
- patches where the object strongly **touches the borders**.

If these categories are too numerous, the model may see too much background or too many truncated objects, and not enough well-centered informative examples.

### 12.1. Start from clean positive patches

We first keep the positive patches that are considered clean, meaning patches with meaningful foreground content and without the strongest geometric ambiguities.

If the number of such patches is

$$
n_{pc} = \mathrm{pos\_clean.sum()}
$$

then this number becomes the reference used to define the caps for the other categories.

### 12.2. Cap the number of touching patches

Touching-border patches are useful because they expose the model to partial objects, but too many of them would overrepresent truncated shapes.

Their number is limited using the chosen touching fraction `touch_frac`:

$$
n_{pt} = \left\lfloor n_{pc}\,\frac{\mathrm{touch\_frac}}{1-\mathrm{touch\_frac}} \right\rfloor
$$

This gives a controlled number of touching examples relative to the clean positive ones.

We also exclude the most problematic cases, such as patches touching both opposite sides, because these often correspond to strongly truncated objects that no longer provide a well-localized shape for training.

### 12.3. Define the total positive reference

The total positive reference after adding the allowed touching examples is

$$
n_p = n_{pc} + n_{pt}
$$

This value is then used to define how many empty and almost empty patches can be added.

### 12.4. Cap empty and almost empty patches

Empty and almost empty patches are still useful because the model must learn background and weak-object situations, but they should not dominate the dataset.

If `empty_frac` is the target fraction of empty patches and `alm_empty` the target fraction of almost empty patches, then the allowed numbers are:

$$
n_{em} =
\left\lfloor
n_p
\frac{\mathrm{empty\_frac} + \mathrm{alm\_empty}}
{1-\mathrm{empty\_frac}-\mathrm{alm\_empty}}
\frac{\mathrm{empty\_frac}}
{\mathrm{empty\_frac}+\mathrm{alm\_empty}}
\right\rfloor
$$

and

$$
n_{ae} =
\left\lfloor
n_p
\frac{\mathrm{empty\_frac} + \mathrm{alm\_empty}}
{1-\mathrm{empty\_frac}-\mathrm{alm\_empty}}
\frac{\mathrm{alm\_empty}}
{\mathrm{empty\_frac}+\mathrm{alm\_empty}}
\right\rfloor
$$

So:

- $n_{em}$ is the number of **empty** patches kept,
- $n_{ae}$ is the number of **almost empty** patches kept.

### Why do this?

This balancing step keeps:

- enough positive examples,
- some touching-border cases,
- some empty and almost empty patches,

but prevents the final training set from being dominated by low-information samples.

In [ ]:
### split dataset into training and test_val
### test val later will be split by the magnification and patch multipliers and separated into two equal sets
### train set will be weighted so that it will have 1% of funcionally empty 5% of almost empty masks
### and masks touching borders will be capped by 20% from whole positive subset

seed = 25

touch_frac = 0.2
empty_frac = 0.1
alm_empty = 0.1

### touching tolerance tells how much particles can touch out of how many
touch_tol = 1/4

df.reset_index(inplace = True, drop = True)

### Get rid of touching both sides patches since such particles are too large for the efficient recogintion
df = df[df['Touch_both_sides'].astype(int) == 0]
df['Touching_NPs'] = ((df['Touch_border'].astype(int)/df['N_NPs'].astype(int))>=touch_tol)

df['Usage'] = np.nan

df_train = df[~df['File_name'].isin(test_val)].copy()

df_weight_tr = pd.DataFrame([])

sdf_gr = df_train.groupby(['NP_size', 'Magnification', 'Patch_mult'])

for key in tqdm(sdf_gr.groups.keys()):
    
    sdf_proc = sdf_gr.get_group(key)
    
    rs = seed + hash(key) % 10_000
    
    pos_clean = ((~sdf_proc['Empty']) & (~sdf_proc['Almost_Empty']) & (~sdf_proc['Touching_NPs']))
    pos_touch = ((~sdf_proc['Empty']) & (~sdf_proc['Almost_Empty']) & (sdf_proc['Touching_NPs']))
    empty = sdf_proc['Empty']
    almost_empty = sdf_proc['Almost_Empty']
    
    n_pc = pos_clean.sum()
    n_pt = int(n_pc * touch_frac / (1 - touch_frac))
    
    n_p = n_pc+n_pt
    n_em = int(n_p * (empty_frac + alm_empty) / (1 - empty_frac - alm_empty) * (empty_frac / (empty_frac+alm_empty)))
    n_ae = int(n_p * (empty_frac + alm_empty) / (1 - empty_frac - alm_empty) * (alm_empty / (empty_frac+alm_empty)))
    
    sub_train = sdf_proc[pos_clean]
    
    if n_pt <= pos_touch.sum():
        pt_kept = sdf_proc[pos_touch].sample(n=n_pt, random_state=rs)
        sub_train = pd.concat([sub_train, pt_kept])
    else:
        sub_train = pd.concat([sub_train, sdf_proc[pos_touch]])
    
    if n_em <= empty.sum():
        em_kept = sdf_proc[empty].sample(n=n_em, random_state=rs)
        sub_train = pd.concat([sub_train, em_kept])
    else:
        sub_train = pd.concat([sub_train, sdf_proc[empty]])
        
    if n_ae <= almost_empty.sum():
        aem_kept = sdf_proc[almost_empty].sample(n=n_ae, random_state=rs)
        sub_train = pd.concat([sub_train, aem_kept])
    else:
        sub_train = pd.concat([sub_train, sdf_proc[almost_empty]])
    
    sub_train['Plot'] = 1
    
    if df_weight_tr.size == 0:
        df_weight_tr = sub_train[['Plot']]
    else:
        df_weight_tr = pd.concat([df_weight_tr, sub_train[['Plot']]])

df_train = df_train.merge(df_weight_tr, left_index=True, right_index=True, how='outer')
df_train.loc[df_train['Plot'] == 1, 'Usage'] = 'tr'

## 13. Build the validation and test subsets

After defining the training set, we construct the validation and test sets from the images that were reserved earlier.

The same general balancing principle is used here as for the training subset:

- we keep control over the proportions of empty, almost empty, touching, and positive patches,
- but now we apply it only inside the subset of images assigned to validation and test.

The important difference is that the validation and test sets are not designed to be as large as possible.  
Instead, their sizes are defined relative to the training set.

In practice, the numbers of validation and test patches are deduced so that they represent chosen fractions of the training-set size.  
This means that even if a reserved image contains many possible candidate patches, we may use only part of them.

This keeps validation and test sets:

- large enough to be representative,
- but still controlled in size,
- and consistent with the overall balance chosen for the training workflow.

In [ ]:
### split dataset into training and test_val
### test val later will be split by the magnification and patch multipliers and separated into two equal sets
### train set will be weighted so that it will have 1% of funcionally empty 5% of almost empty masks
### and masks touching borders will be capped by 20% from whole positive subset

seed = 25

touch_frac = 0.2
empty_frac = 0.1
alm_empty = 0.1

### Find the number of training images in order to limit number of validation and test instances
tr_s = df_train['Plot'].sum()

val_frac = 0.2
test_frac = 0.2

n_vt = int(tr_s * (val_frac + test_frac) / (1 - val_frac - test_frac))


df_test = df[df['File_name'].isin(test_val)].copy()

df_weight_tr = pd.DataFrame([])

sdf_gr = df_test.groupby(['NP_size', 'Magnification', 'Patch_mult'])

n_sub_vt = int(n_vt/len(sdf_gr.groups.keys()))

for key in tqdm(sdf_gr.groups.keys()):
    
    sdf_proc = sdf_gr.get_group(key)
    
    ### Define the randomisation seed to introduce the repetability to the split of data
    rs = seed + hash(key) % 10_000
    
    pos_clean = ((~sdf_proc['Empty']) & (~sdf_proc['Almost_Empty']) & (~sdf_proc['Touching_NPs']))
    pos_touch = ((~sdf_proc['Empty']) & (~sdf_proc['Almost_Empty']) & (sdf_proc['Touching_NPs']))
    empty = sdf_proc['Empty']
    almost_empty = sdf_proc['Almost_Empty']
    
    ### Number of positive validatio nand test sets are capped by 30 to avoid too few positive expampels
    n_pc = int(max([30, n_sub_vt * pos_clean.sum()/pos_clean.shape[0]]))
    n_pt = int(max([30, n_sub_vt * pos_touch.sum()/pos_touch.shape[0]]))
    n_em = int(n_sub_vt * empty.sum()/empty.shape[0])
    n_ae = int(n_sub_vt * almost_empty.sum()/almost_empty.shape[0])
    
    ### If overall amount of positive examples are less than one we want to have
    ### We'll use them all
    if n_pc <= pos_clean.sum():
        pc_kept = sdf_proc[pos_clean].sample(n=n_pc, random_state=rs)
    else:
        pc_kept = sdf_proc[pos_clean].sample(n=pos_clean.sum(), random_state=rs)
    pc_n = pc_kept.shape[0]
    sub_val = pc_kept.iloc[:int(pc_n/2)]
    sub_test = pc_kept.iloc[int(pc_n/2):]
    
    if n_pt <= pos_touch.sum():
        pt_kept = sdf_proc[pos_touch].sample(n=n_pt, random_state=rs)
    else:
        pt_kept = sdf_proc[pos_touch].sample(n=pos_touch.sum(), random_state=rs)
    pt_n = pt_kept.shape[0]
    sub_val = pd.concat([sub_val, pt_kept.iloc[:int(pt_n/2)]])
    sub_test = pd.concat([sub_test, pt_kept.iloc[int(pt_n/2):]])
    
    if n_em <= empty.sum():
        em_kept = sdf_proc[empty].sample(n=n_em, random_state=rs)
    else:
        em_kept = sdf_proc[empty].sample(n=empty.sum(), random_state=rs)
    em_n = em_kept.shape[0]
    sub_val = pd.concat([sub_val, em_kept.iloc[:int(em_n/2)]])
    sub_test = pd.concat([sub_test, em_kept.iloc[int(em_n/2):]])
    
    if n_ae <= almost_empty.sum():
        aem_kept = sdf_proc[almost_empty].sample(n=n_ae, random_state=rs)
    else:
        aem_kept = sdf_proc[almost_empty].sample(n=almost_empty.sum(), random_state=rs)
    ae_n = aem_kept.shape[0]
    sub_val = pd.concat([sub_val, aem_kept.iloc[:int(ae_n/2)]])
    sub_test = pd.concat([sub_test, aem_kept.iloc[int(ae_n/2):]])
    
    sub_val['Usage'] = 'val'
    sub_val['Plot'] = 1
    
    sub_test['Usage'] = 'test'
    sub_test['Plot'] = 1
    
    sub_val_test = pd.concat([sub_val, sub_test])
    
    if df_weight_tr.size == 0:
        df_weight_tr = sub_val_test[['Plot', 'Usage']]
    else:
        df_weight_tr = pd.concat([df_weight_tr, sub_val_test[['Plot', 'Usage']]])

df_test.drop(columns = 'Usage', inplace = True)
df_test = df_test.merge(df_weight_tr, left_index=True, right_index=True, how='outer')

## 14. Check the integrity of the split

Before saving anything, we verify that the selected entries are unique.

This is a simple but important safety check: each patch description should appear only once in the final dataset table.

In [ ]:
df_test.index.drop_duplicates().shape == df_test.index.shape

In [ ]:
df_train.index.drop_duplicates().shape == df_train.index.shape

## 15. Save the final patch table

At this point, the train, validation, and test selections are defined.

We merge them back into one dataframe and save it to disk.  
This file becomes the formal description of the dataset.

Keeping this table is useful because it records:

- where each patch comes from,
- how it was classified,
- which split it belongs to.

This makes the whole preparation pipeline reproducible.

In [ ]:
df = pd.concat([df_train, df_test])

df.sort_index(inplace = True)

### Overwrite previously saved training patch table
df.to_csv(pjoin(saving_path, f'Whole Training set {base_patch_size}.csv'), sep = '\t')

## 16. Generate and save the final image and mask patches

So far, we have only described the selected patches in a dataframe.  
Now we turn that description into real files that can be used for training.

For each selected entry, we:

1. load the original image and its mask,
2. crop the region defined in the dataframe,
3. resize it to a common patch size,
4. save both the image patch and the corresponding mask patch.

The dataframe acts as the source of truth here: only the patches that were explicitly selected for training, validation, or testing are exported. This keeps the saved dataset fully consistent with the split designed above.

It is also important to use different interpolation methods for images and masks during resizing.

For the **image**, we use a smooth interpolation method because the pixel values represent measured intensity. Smooth interpolation preserves visual continuity and avoids blocky artifacts.

For the **mask**, we use **nearest-neighbor interpolation** because the mask contains discrete labels, not continuous values. A segmentation mask should stay binary or categorical after resizing. If we used a smooth interpolation for masks, it would create artificial intermediate values at the boundaries, which would corrupt the labels and make the training targets incorrect.

In [ ]:
### Based on the dataframe above prepare the patches for the dataset
### use only patches listed as train, val or test, no need to make more patches

base_patch_size = 512

mask_save_path = pjoin(root, f'Training {base_patch_size} TP', 'Masks')
img_save_path = pjoin(root, f'Training {base_patch_size} TP', 'Images')

if not os.path.exists(mask_save_path):
    os.makedirs(mask_save_path)
else:
    pass

if not os.path.exists(img_save_path):
    os.makedirs(img_save_path)
else:
    pass

assert (df.index.drop_duplicates().shape == df.index.shape)

df[~df['Usage'].isna()]['File_name']
df['Patch_name'] = np.nan

for fn in tqdm(df[~df['Usage'].isna()]['File_name'].unique()):
    
    df_fn = df[(~df['Usage'].isna()) & (df['File_name']==fn)]
    
    mask_path = np.array([i for i in masks if fn in i])
    img_path = np.array([i for i in images if fn[:-5] in i])
    
    assert len(mask_path) == 1
    assert len(img_path) == 1
    
    with TiffFile(mask_path.item()) as tif:
        mask = tif.asarray()
        ### binarize the mask
        mask_bin = (mask.sum(axis = 2)>0).astype(int)

    with TiffFile(img_path.item()) as tif:
        img = tif.asarray()
    
    for loc_idx in tqdm(df_fn.index):
    
        df_patch = df_fn.loc[loc_idx]
        
        patch_mult = df_patch['Patch_mult']
        w_st = df_patch['Patch_x_pos']
        h_st = df_patch['Patch_y_pos']
        
        saving_file_name = fn[:-5] + f'_pm{patch_mult}_wst{w_st}_hst{h_st}'
                
        ps = base_patch_size*patch_mult

        ### make mask patches
        mask_patch = mask_bin[int(h_st):int(h_st+ps), int(w_st):int(w_st+ps)]
        
        mask_image = Image.fromarray(mask_patch)
        
        mask_image = mask_image.resize((base_patch_size,base_patch_size),
                                       resample = Image.NEAREST)
        
        mask_patch_res = np.array(mask_image)

        ### make img patches
        img_patch = img[int(h_st):int(h_st+ps), int(w_st):int(w_st+ps)]

        img_image = Image.fromarray(img_patch)
        img_image = img_image.resize((base_patch_size,base_patch_size),
                                     resample = Image.LANCZOS)

        img_patch_res = np.array(img_image)
        
        tifffile.imwrite(pjoin(mask_save_path, saving_file_name + ".tiff"), mask_patch_res.astype(np.uint8))
        tifffile.imwrite(pjoin(img_save_path, saving_file_name + ".tiff"), img_patch_res.astype(np.uint8))
        
        df.loc[loc_idx, 'Patch_name'] = saving_file_name

saving_path = pjoin(root, f'Training {base_patch_size} TP')
if not os.path.exists(saving_path):
    os.makedirs(saving_path)
else:
    pass

df.to_csv(pjoin(saving_path, f'Whole Training set {base_patch_size}.csv'), sep = '\t')

## 17. Visualize a few representative patch categories

Before moving on, it is useful to look at a few concrete examples from the dataset we just saved.

Here we reload the saved dataframe, select one patch from each of three categories:

- **empty**: no annotated particle in the patch,
- **almost empty**: only a very small amount of positive mask is present,
- **not empty**: a patch containing a clear positive object.

For each category, we display the **image patch** and its **mask** one above the other.

This is a simple but important sanity check. It helps us verify that:

- the dataframe labels match the actual visual content,
- the saved image and mask files correspond correctly,
- the dataset contains the patch types we expect.

In [ ]:
base_patch_size = 512
saving_path = pjoin(root, f"Training {base_patch_size} TP")
csv_path = pjoin(saving_path, f"Whole Training set {base_patch_size}.csv")

img_save_path = pjoin(saving_path, "Images")
mask_save_path = pjoin(saving_path, "Masks")

df_saved = pd.read_csv(csv_path, sep="\t", index_col=0)

# Keep only rows that were actually exported
df_saved = df_saved[~df_saved["Usage"].isna()].copy()

# Select one example for each category
examples = {
    "Empty": df_saved[df_saved["Empty"] == True].sample().iloc[0],
    "Almost empty": df_saved[(df_saved["Almost_Empty"] == True) & (df_saved["Empty"] == False)].sample().iloc[0],
    "Not empty": df_saved[(df_saved["Empty"] == False) & (df_saved["Almost_Empty"] == False)].sample().iloc[0],
}

fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(12, 8),
    constrained_layout=True
)

for col, (label, row) in enumerate(examples.items()):
    patch_name = row["Patch_name"]

    img = tifffile.imread(pjoin(img_save_path, patch_name + ".tiff"))
    mask = tifffile.imread(pjoin(mask_save_path, patch_name + ".tiff"))

    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(label)
    axes[0, col].axis("off")

    axes[1, col].imshow(mask, cmap="gray")
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Image", fontsize=12)
axes[1, 0].set_ylabel("Mask", fontsize=12)

plt.show()

## 18. Final sanity check of the exported dataset

As a last step, we summarize the saved subset by category.

This gives us a quick overview of how the final dataset is distributed across:

- nanoparticle type,
- magnification,
- empty and non-empty patches,
- dataset usage split.

This final check helps confirm that the exported training data matches the intended design.

In [ ]:
df[df['Plot']==1].groupby(['NP_size', 'Magnification', 'Empty', 'Usage']).apply(lambda x: pd.Series({
    'Number_of_instances' : (x.shape[0]),
})).reset_index()

## What we achieved in this notebook

We started from raw microscopy images and annotation files, and we transformed them into a structured dataset ready for model training.

More precisely, we:

- extracted masks from annotation files,
- matched masks with source images,
- defined candidate patches at multiple scales,
- measured patch properties,
- built train, validation, and test splits without leakage,
- exported the final image and mask patches.

The next step will be to load this dataset into PyTorch and use it for segmentation model training.